In [1]:
import os
from src.rag.ingestion.reader import load_documents_from_path
from src.rag.ingestion.ingestion import run_ingestion
from src.rag.ingestion.indexer import create_hierarchical_index
from llama_index.core.retrievers import AutoMergingRetriever
from llama_index.core import Settings
from llama_index.embeddings.openai import OpenAIEmbedding
# Uncomment and set this if you are using default OpenAI embeddings for indexing
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"
Settings.embed_model = OpenAIEmbedding(
    model_name="nomic-embed-text-v1.5",  # Replace with the exact identifier loaded in LM Studio
    api_base="http://localhost:1234/v1",  # LM Studio default server URL
    api_key="lm-studio"                   # Dummy key to satisfy client library
)
def test_retrieval_only():
    print("1. Loading Documents...")
    # Point this to your specific data folder
    data_path = r"E:/College Projects/llamaIndex/AcademicSecondBrain/src/data"
    documents = load_documents_from_path(data_path)
    print(f"   Loaded {len(documents)} document objects.")

    print("\n2. Running Ingestion Pipeline (Cleaning & Chunking)...")
    all_nodes, leaf_nodes = run_ingestion(documents)
    print(f"   Generated {len(all_nodes)} total nodes in the hierarchy.")
    print(f"   Extracted {len(leaf_nodes)} leaf nodes for the vector store.")

    print("\n3. Building ChromaDB Index...")
    index = create_hierarchical_index(all_nodes, leaf_nodes, db_path="./chroma_db")
    print("   Index built successfully!")

    print("\n4. Testing Retrieval (Bypassing LLM)...")
    # Initialize the retrievers directly from the index
    base_retriever = index.as_retriever(similarity_top_k=12)
    retriever = AutoMergingRetriever(
        base_retriever,
        storage_context=index.storage_context
    )

    test_question = "What is the main topic of these documents?"
    print(f"   Q: {test_question}")

    # .retrieve() ONLY fetches nodes, it does not call an LLM to generate text
    retrieved_nodes = retriever.retrieve(test_question)

    print(f"\n   Successfully retrieved {len(retrieved_nodes)} nodes!")

    print("\n--- Retrieved Chunks ---")
    for i, node_with_score in enumerate(retrieved_nodes):
        # Extract the underlying BaseNode object
        node = node_with_score.node

        print(f"\nSource {i+1} | Length: {len(node.text)} chars | Score: {node_with_score.score:.3f}")

        # Print a preview of the text to verify your cleaning and merging worked
        preview = node.text[:200].replace("\n", " ")
        print(f"Preview: {preview}...")

if __name__ == "__main__":
    test_retrieval_only()

1. Loading Documents...
   Loaded 10 document objects.

2. Running Ingestion Pipeline (Cleaning & Chunking)...


AttributeError: property 'text' of 'Document' object has no setter